# 04_BL_Walkforward — Black-Litterman 실험 실행

**역할**: bl_config.py에 정의된 모든 실험을 walk-forward로 실행하고 결과를 `results/` 폴더에 저장.

## 실행 순서
1. 데이터 로드 (panel, daily_ret)
2. LSTM 예측 로드 (`bl_runner.load_lstm_pred`)
3. monthly_cache 빌드 (`bl_runner.build_monthly_cache`) — 모든 슬롯 공유
4. 전체 실험 walk_forward 실행 → pkl 저장

> Dispatcher (`get_vol_series`, `get_prior_weights`, `get_Q`, `get_omega`) 와 `walk_forward` 본체는 [`bl_runner.py`](bl_runner.py) 에 정의됨.
> 분석/시각화는 **05b_Analyze.ipynb** 에서.

In [1]:
import pandas as pd
import numpy as np
import pickle
import warnings
from pathlib import Path

import matplotlib
matplotlib.use('Agg')

warnings.filterwarnings('ignore')

# BL 백테스트 엔진 (dispatcher + walk_forward 본체)
from bl_runner import load_lstm_pred, build_monthly_cache, walk_forward
from bl_config import EXPERIMENTS, BASELINE

BASE_DIR    = Path.cwd()
DATA_DIR    = BASE_DIR / 'data'
RESULTS_DIR = BASE_DIR / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

assert DATA_DIR.exists(), f'data/ 폴더 없음. 01_DataCollection.ipynb 먼저 실행하세요.\n경로: {DATA_DIR}'

# ── 공통 파라미터 ──────────────────────────────────────────────
TRAIN_WINDOW  = 60
THRESH_DAILY  = 0.9
TAU           = 0.1
PCT_GROUP     = 0.30
START_PRED    = '2010-01-01'

print(f'설정 완료')
print(f'  DATA_DIR    : {DATA_DIR}')
print(f'  RESULTS_DIR : {RESULTS_DIR}')
print(f'  실험 수     : {len(EXPERIMENTS)}개')

설정 완료
  DATA_DIR    : /Users/yoonseokim/2025_main_bootcamp/4th_final_project/finance_project/final_pt/data
  RESULTS_DIR : /Users/yoonseokim/2025_main_bootcamp/4th_final_project/finance_project/final_pt/results
  실험 수     : 90개


In [2]:
# ── 데이터 로드 ───────────────────────────────────────────────
panel     = pd.read_csv(DATA_DIR / 'monthly_panel.csv', parse_dates=['date'])
panel     = panel.set_index(['date', 'ticker'])
daily_ret = pd.read_pickle(DATA_DIR / 'daily_returns.pkl')

all_dates  = panel.index.get_level_values('date').unique().sort_values()
pred_dates = all_dates[all_dates >= START_PRED]

spy_series = panel['spy_ret'].groupby(level='date').first()
rf_series  = panel['rf_1m'].groupby(level='date').first()

print(f'패널       : {panel.shape}')
print(f'일별 수익률: {daily_ret.shape}')
print(f'예측 기간  : {pred_dates[0].date()} ~ {pred_dates[-1].date()} ({len(pred_dates)}개월)')

패널       : (103878, 13)
일별 수익률: (5595, 824)
예측 기간  : 2010-01-31 ~ 2025-12-31 (192개월)


In [3]:
# ── LSTM 예측값 로드 (없으면 None) ────────────────────────────
# bl_config.py의 BASELINE['lstm_pred_path']에 경로 지정.
# 파일이 없으면 모든 실험 자동 skip (현재는 LSTM 단일 옵션).
_lstm_path = Path(BASELINE.get('lstm_pred_path', ''))
lstm_state = load_lstm_pred(_lstm_path, pred_dates)
LSTM_AVAILABLE = lstm_state['available']

if LSTM_AVAILABLE:
    print(f'LSTM 예측 로드: {len(lstm_state["preds"]):,}행')
    print(f'LSTM 월별 피벗: {lstm_state["monthly"].shape[0]}개월 × {lstm_state["monthly"].shape[1]}종목')
else:
    print(f'⚠  LSTM 예측 파일 없음 → 모든 실험 스킵')
    print(f'   경로: {_lstm_path}')

LSTM 예측 로드: 2,498,243행
LSTM 월별 피벗: 192개월 × 614종목


## Dispatcher 함수

Dispatcher (`get_vol_series`, `get_prior_weights`, `get_Q`, `get_omega`) 와 `walk_forward` 본체는 **[`bl_runner.py`](bl_runner.py)** 에 정의됨.

본 노트북은 import 만 하고, 05b_Analyze.ipynb 의 sensitivity sweep 도 같은 모듈을 재사용.


In [4]:
# ── 월별 공유 데이터 캐시 빌드 (Sigma, π 컴포넌트) ───────────────
# 모든 실험에서 동일하므로 1회만 계산. 실험별로 달라지는 것:
#   vol_series(p_mode), P(p_weight), Q, omega, w → walk_forward 안에서 처리.
monthly_cache = build_monthly_cache(
    panel=panel,
    daily_ret=daily_ret,
    pred_dates=pred_dates,
    all_dates=all_dates,
    spy_series=spy_series,
    rf_series=rf_series,
    train_window=TRAIN_WINDOW,
    thresh_daily=THRESH_DAILY,
    verbose=True,
)
print('이후 실험은 Sigma 재계산 없이 캐시에서 로드')

  캐시 2010-01 (1/192, 1%) | 경과 0.0분 | ETA 0.0분
  캐시 2012-01 (25/192, 13%) | 경과 0.0분 | ETA 0.1분
  캐시 2014-01 (49/192, 26%) | 경과 0.0분 | ETA 0.1분
  캐시 2016-01 (73/192, 38%) | 경과 0.0분 | ETA 0.1분
  캐시 2018-01 (97/192, 51%) | 경과 0.1분 | ETA 0.1분
  캐시 2020-01 (121/192, 63%) | 경과 0.1분 | ETA 0.0분
  캐시 2022-01 (145/192, 76%) | 경과 0.1분 | ETA 0.0분
  캐시 2024-01 (169/192, 88%) | 경과 0.1분 | ETA 0.0분

캐시 완료: 192/192개월 | 0.1분
이후 실험은 Sigma 재계산 없이 캐시에서 로드


## walk_forward 실행

`bl_runner.walk_forward(cfg, monthly_cache, pred_dates, lstm_state, spy_series, ...)` 를 호출하여 슬롯별 백테스트.

결과 dict: `config`, `ret` (net), `gross_ret`, `spy_ret`, `weights`, `comp`, `meta`, `errors`.


In [5]:
# ══════════════════════════════════════════════════════════════
# 전체 실험 실행 + 저장
# ══════════════════════════════════════════════════════════════
import time
SKIP_IF_EXISTS = True   # True → 이미 저장된 실험은 재실행 않고 스킵

completed, skipped = [], []
_all_t0 = time.time()

# LSTM csv 없으면 모든 실험 스킵 (모든 슬롯이 LSTM 예측 변동성에 의존)
if not LSTM_AVAILABLE:
    skipped = [cfg['name'] for cfg in EXPERIMENTS]
    print(f'⚠ LSTM 예측 파일 없음 → 전체 {len(EXPERIMENTS)}개 실험 모두 스킵')
else:
    run_list = [cfg for cfg in EXPERIMENTS
                if not (SKIP_IF_EXISTS and (RESULTS_DIR / f"{cfg['name']}.pkl").exists())]
    skipped  = [cfg['name'] for cfg in EXPERIMENTS
                if SKIP_IF_EXISTS and (RESULTS_DIR / f"{cfg['name']}.pkl").exists()]

    print(f'실행 예정: {len(run_list)}개 / 전체 {len(EXPERIMENTS)}개')
    print(f'스킵 (기존 결과): {len(skipped)}개\n')

    for cfg in run_list:
        name     = cfg['name']
        out_path = RESULTS_DIR / f'{name}.pkl'

        done_so_far = len(completed)
        remaining   = len(run_list) - done_so_far
        elapsed_all = time.time() - _all_t0
        avg_per_exp = elapsed_all / done_so_far if done_so_far > 0 else 0
        eta_all     = avg_per_exp * remaining

        print(f'\n[{done_so_far+1}/{len(run_list)}] {name}  |  '
              f'전체 경과 {elapsed_all/60:.1f}분  |  ETA {eta_all/60:.1f}분')

        result = walk_forward(
            cfg, monthly_cache, pred_dates, lstm_state,
            spy_series=spy_series, tau=TAU, pct_group=PCT_GROUP, verbose=True,
        )

        with open(out_path, 'wb') as f:
            pickle.dump(result, f)

        completed.append(name)

total_elapsed = (time.time() - _all_t0) / 60
print(f'\n{"=" * 60}')
print(f'완료: {len(completed)}개 / 스킵: {len(skipped)}개 / 총 {total_elapsed:.1f}분')
print(f'저장 위치: {RESULTS_DIR}')

실행 예정: 0개 / 전체 90개
스킵 (기존 결과): 90개


완료: 0개 / 스킵: 90개 / 총 0.0분
저장 위치: /Users/yoonseokim/2025_main_bootcamp/4th_final_project/finance_project/final_pt/results
